In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps git+https://github.com/rupaut98/unsloth.git@fix-cpt-error

In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install unsloth

In [2]:
# Add after imports but before model loading
import torch.amp.grad_scaler
from unsloth import is_bfloat16_supported

# Only patch if needed (FP16-only hardware)
def patch_grad_scaler_if_needed(model=None, target_modules=None):
    """Conditionally patch PyTorch's GradScaler based on hardware and model configuration"""
    # Check if we're on hardware without BF16 support
    if not is_bfloat16_supported():
        # Skip patching for Gemma-3 models
        is_gemma3 = False
        if model is not None:
            # Check model name or configuration for "gemma-3"
            model_name = getattr(model, "name_or_path", "")
            if not model_name and hasattr(model, "config"):
                model_name = getattr(model.config, "name_or_path", "")
                if not model_name and hasattr(model.config, "_name_or_path"):
                    model_name = model.config._name_or_path
            is_gemma3 = "gemma-3" in str(model_name).lower()

        if is_gemma3:
            print("Unsloth: Detected Gemma-3 model, skipping GradScaler patch")
            return False

        # Check if we're training embedding layers (either from arguments or manually check)
        train_embeddings = False
        if target_modules is not None:
            train_embeddings = "embed_tokens" in target_modules or "lm_head" in target_modules
        elif model is not None:
            # Look through model parameters for embedding layers
            for name, _ in model.named_parameters():
                if "embed_tokens" in name or "lm_head" in name:
                    train_embeddings = True
                    break

        if train_embeddings:
            # Only patch if we're training embedding layers on FP16-only hardware
            original_unscale_grads = torch.amp.grad_scaler.GradScaler._unscale_grads_

            def patched_unscale_grads(self, optimizer, inv_scale, found_inf, allow_fp16=False):
                return original_unscale_grads(self, optimizer, inv_scale, found_inf, True)

            # Apply the patch
            torch.amp.grad_scaler.GradScaler._unscale_grads_ = patched_unscale_grads
            print("Unsloth: Patched GradScaler to allow FP16 gradients for embedding training")
            return True

    return False

# Call the function with your target modules before model creation
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj",
                 "embed_tokens", "lm_head"]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
# Prompt that sets up the context for an equation
prompt = "What are Euler’s and Lagrange’s central configurations for three bodies?"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 What are Euler’s and Lagrange’s central configurations for three bodies? Euler’s central configurations for three bodies are those in which the three bodies lie on a straight line, with the two outer bodies being equal and opposite in mass, and the inner body being twice as massive as the outer bodies. This configuration is named after Leonhard Euler, who first described it in the 18th century.

Lagrange’s central configurations for three bodies are those in which the three bodies form an equilateral triangle. This configuration is named after Joseph-Louis Lagrange, who first described it in the 18th century. In this configuration, the three bodies are in a state of equilibrium, with each body being attracted to the other two bodies with equal force.<|endoftext|>


In [5]:
# Prompt that sets up the context for an equation
prompt = "Why are numerical methods often necessary for solving the n-body problem when n>3?"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Why are numerical methods often necessary for solving the n-body problem when n>3? The n-body problem is a classical problem in physics and astronomy that involves predicting the motion of a group of celestial bodies interacting with each other gravitationally. When the number of bodies \( n \) is greater than 3, the problem becomes extremely complex and cannot be solved analytically using traditional mathematical methods. This is because the equations of motion for the n-body problem are nonlinear and coupled, making it impossible to find a closed-form solution.

Numerical methods are often necessary to solve the n-body problem when \( n > 3 \) because:

1. **Complexity of the Problem**: The equations of motion for the n-body problem are highly nonlinear and coupled, making it difficult to find an analytical solution. Numerical methods allow us to approximate the solution by breaking down the problem into smaller, more manageable parts.

2. **Computational Power**: Nu

In [6]:
# Prompt that sets up the context for an equation
prompt = "Write the LaTeX code for the equations of motion for the n-body problem using Newton's law of gravitation."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Write the LaTeX code for the equations of motion for the n-body problem using Newton's law of gravitation. The equations of motion for the n-body problem using Newton's law of gravitation are given by:

\begin{align*}
\frac{d^2\mathbf{r}_i}{dt^2} &= -G \sum_{j=1, j \neq i}^n \frac{m_j (\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}, \quad i = 1, 2, \ldots, n
\end{align*}

where $\mathbf{r}_i$ is the position vector of the $i$-th body, $m_i$ is its mass, $G$ is the gravitational constant, and $t$ is time. The sum is taken over all the other bodies in the system, excluding the $i$-th body itself. The term $\mathbf{r}_j - \mathbf{r}_i$ represents the vector pointing from the $i$-th body to the $j$-th body, and $|\mathbf{r}_j - \mathbf{r}_i|$ is its magnitude. The factor of $G$ is included to ensure that the units of the equation are consistent with the units of the other quantities.<|endoftext|>


In [7]:
# Prompt that sets up the context for an equation
prompt = "Implement SymPy code to verify whether three given masses at specific positions form a central configuration."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Implement SymPy code to verify whether three given masses at specific positions form a central configuration. The masses are \( m_1 = 1 \), \( m_2 = 2 \), and \( m_3 = 3 \). The positions of the masses are given by \( \mathbf{r}_1 = (0, 0) \), \( \mathbf{r}_2 = (1, 0) \), and \( \mathbf{r}_3 = (0, 1) \). Determine if the configuration is central by checking if the following equation holds:

$$
\sum_{i=1}^3 \frac{m_i}{r_{ij}^3} \mathbf{r}_{ij} = \mathbf{0}
$$

where \( \mathbf{r}_{ij} = \mathbf{r}_i - \mathbf{r}_j \) and \( r_{ij} = \|\mathbf{r}_{ij}\| \).
To determine if the given configuration is central, we need to verify the equation:

$$
\sum_{i=1}^3 \frac{m_i}{r_{ij}^3} \mathbf{r}_{ij} = \mathbf{0}
$$

First, we calculate the distances \( r_{ij} \) between each pair of masses:

1. \( r_{12} = \|\mathbf{r}_1 - \mathbf{r}_2\| = \|(0, 0) - (1, 0)\| = 1 \)
2. \( r_{13} = \|\mathbf{r}_1 - \mathbf{r}_3\| = \|(0, 0) - (0, 1)\| = 1 \)
3. \( r_{23} = \|\mathbf{r}_2 - \math

In [8]:
# Prompt that sets up the context for an equation
prompt = "Write LaTeX to describe how central configurations are defined mathematically. Then implement SymPy code to find all possible central configurations for three equal masses in 2D space."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Write LaTeX to describe how central configurations are defined mathematically. Then implement SymPy code to find all possible central configurations for three equal masses in 2D space. To define central configurations mathematically, we need to understand the concept of a central configuration in the context of the n-body problem. A central configuration is a special arrangement of the masses such that the acceleration vector of each mass is proportional to the position vector of that mass relative to the center of mass of the system. Mathematically, for a system of \( n \) masses \( m_1, m_2, \ldots, m_n \) in \( \mathbb{R}^d \), a central configuration is a configuration where there exists a constant \( \lambda \) such that for each \( i \),

\[
\sum_{j=1, j \neq i}^n \frac{m_j}{r_{ij}^3} \mathbf{r}_{ij} = \lambda \mathbf{r}_i,
\]

where \( \mathbf{r}_i \) is the position vector of the \( i \)-th mass, \( r_{ij} = \|\mathbf{r}_i - \mathbf{r}_j\| \) is the distance be

In [5]:
# Prompt that sets up the context for an equation
prompt = "Singularities in an n‑body system occur when two or more bodies come extremely close to each other, causing the gravitational forces (and hence accelerations) to approach infinity. Please provide a clear LaTeX explanation showing how such singularities (specifically collision singularities) appear in the equations of motion derived from Newton’s laws. Then, write complete Python code using the SymPy library to simulate a near‑collision scenario for three bodies in 2D space. Use explicit initial conditions for the positions and velocities. In your code, include comments on the numerical challenges encountered during the simulation (such as instability due to large forces near collisions), and describe any potential strategies (such as regularization techniques or adaptive time‑stepping) that could be used to mitigate these issues. Ensure that the code is self-contained and runnable."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=3072, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Singularities in an n‑body system occur when two or more bodies come extremely close to each other, causing the gravitational forces (and hence accelerations) to approach infinity. Please provide a clear LaTeX explanation showing how such singularities (specifically collision singularities) appear in the equations of motion derived from Newton’s laws. Then, write complete Python code using the SymPy library to simulate a near‑collision scenario for three bodies in 2D space. Use explicit initial conditions for the positions and velocities. In your code, include comments on the numerical challenges encountered during the simulation (such as instability due to large forces near collisions), and describe any potential strategies (such as regularization techniques or adaptive time‑stepping) that could be used to mitigate these issues. Ensure that the code is self-contained and runnable. To understand singularities in an n-body system, let's consider the equations of motion 

In [1]:
%%capture
!pip install transformers accelerate sentencepiece

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Define Model ID and Prompt
model_id = "Qwen/Qwen2.5-Math-1.5B" # Base Qwen2.5 7B model
# 2. Load Tokenizer and Model
# Use bfloat16 for efficiency if available, otherwise float16. 'auto' lets transformers choose.
# device_map="auto" helps distribute layers if the model is large for one GPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto", # Use torch.bfloat16 if available, else torch.float16
    device_map="auto"   # Automatically map model layers to available devices (GPU/CPU)
)

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [2]:
# Prompt that sets up the context for an equation
prompt = "What are Euler’s and Lagrange’s central configurations for three bodies?"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 What are Euler’s and Lagrange’s central configurations for three bodies? Euler’s central configurations for three bodies are those where the three bodies lie on a line, and the center of mass of the system is located at the center of the line. This means that the three bodies are aligned in a straight line, and the center of mass of the system is at the midpoint of the line.

Lagrange’s central configurations for three bodies are those where the three bodies lie on a circle, and the center of mass of the system is located at the center of the circle. This means that the three bodies are arranged in a circular pattern, and the center of mass of the system is at the center of the circle.

What is the significance of the paper by Alain Albouy and Richard Moeckel? The paper by Alain Albouy and Richard Moeckel is significant because it provides a complete classification of the central configurations for three bodies. This classification is important because it allows us to 

In [3]:
# Prompt that sets up the context for an equation
prompt = "Why are numerical methods often necessary for solving the n-body problem when n>3?"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Why are numerical methods often necessary for solving the n-body problem when n>3? The n-body problem involves predicting the motion of n celestial bodies interacting with each other through gravitational forces. When n>3, the problem becomes highly complex due to the large number of interactions and the non-linear nature of the gravitational forces. This complexity makes it difficult to find an analytical solution, which is why numerical methods are often necessary.

What is the significance of the n-body problem in the context of celestial mechanics? The n-body problem is significant in celestial mechanics because it helps us understand the motion of celestial bodies in the universe. By studying the n-body problem, we can gain insights into the behavior of stars, planets, and other celestial bodies, which is crucial for understanding the evolution of the universe and the formation of galaxies.

What is the difference between the n-body problem and the three-body prob

In [4]:
# Prompt that sets up the context for an equation
prompt = "Write the LaTeX code for the equations of motion for the n-body problem using Newton's law of gravitation."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Write the LaTeX code for the equations of motion for the n-body problem using Newton's law of gravitation. The equations of motion for the n-body problem using Newton's law of gravitation are given by:

\begin{align*}
\frac{d^2\mathbf{r}_i}{dt^2} &= -G\sum_{j=1, j\neq i}^n \frac{m_j(\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}, \quad i = 1, 2, \ldots, n
\end{align*}

where $\mathbf{r}_i$ is the position vector of the $i$-th body, $m_i$ is its mass, $G$ is the gravitational constant, and the sum is taken over all the other bodies in the system.<|endoftext|>


In [5]:
# Prompt that sets up the context for an equation
prompt = "Implement SymPy code to verify whether three given masses at specific positions form a central configuration."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Implement SymPy code to verify whether three given masses at specific positions form a central configuration. The masses are \( m_1 = 1 \), \( m_2 = 2 \), and \( m_3 = 3 \), and their positions are \( \mathbf{r}_1 = (0, 0) \), \( \mathbf{r}_2 = (1, 0) \), and \( \mathbf{r}_3 = (0, 1) \) respectively. A central configuration is one where the acceleration of each mass is proportional to the force exerted by the other masses, with the proportionality constant being the same for all masses.

Given the potential energy function \( V(\mathbf{r}_1, \mathbf{r}_2, \mathbf{r}_3) = \sum_{i=1}^3 \sum_{j=i+1}^3 \frac{m_i m_j}{\|\mathbf{r}_i - \mathbf{r}_j\|} \), verify if the given configuration is central by checking if the acceleration of each mass is proportional to the force exerted by the other masses.

To verify if the given configuration is central, we need to check if the acceleration of each mass is proportional to the force exerted by the other masses. The force exerted b

In [6]:
# Prompt that sets up the context for an equation
prompt = "Explain and represent mathematically how singularities (e.g., collisions) occur in an n-body system. Write SymPy code to simulate a near-collision scenario for three bodies with initial positions and velocities."

# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance,"
# Or: prompt = "The general form of a quadratic equation is ax^2 + bx + c = 0. For instance, " # Might just output variables

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer.eos_token_id)
generated_text = tokenizer.batch_decode(outputs)[0]
print("Model Response:\n", generated_text)

Model Response:
 Explain and represent mathematically how singularities (e.g., collisions) occur in an n-body system. Write SymPy code to simulate a near-collision scenario for three bodies with initial positions and velocities. Discuss the challenges in accurately representing such collisions in a numerical simulation.
In an n-body system, singularities can occur when two or more bodies come very close to each other, leading to a collision. Mathematically, this can be represented by the equations of motion for the bodies, which are typically derived from Newton's laws of motion and gravitation. When two bodies collide, their positions and velocities become undefined, and the equations of motion break down.

To simulate a near-collision scenario for three bodies with initial positions and velocities, we can use numerical integration methods such as the Runge-Kutta method. However, when two bodies come very close to each other, the forces between them become very large, and the numerica

In [7]:
# Apply conditional patch
patch_applied = patch_grad_scaler_if_needed(model=model, target_modules=target_modules)

Unsloth: Patched GradScaler to allow FP16 gradients for embedding training


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [9]:
from datasets import load_dataset
import os # Import os to check file existence

# Define the specific path to your single data file
data_file_path = "/kaggle/input/book-batch1/cleaned_book_batch1.md"

print(f"Looking for data file at: {data_file_path}")

# --- Optional but recommended: Check if the file actually exists ---
if not os.path.exists(data_file_path):
    # If it doesn't exist, list the contents of the input directory for debugging
    input_dir = "/kaggle/input/"
    print(f"Error: Data file not found at {data_file_path}")
    print(f"Listing contents of {input_dir}:")
    try:
        print(os.listdir(input_dir))
        if os.path.exists(os.path.dirname(data_file_path)):
             print(f"Listing contents of {os.path.dirname(data_file_path)}:")
             print(os.listdir(os.path.dirname(data_file_path)))
    except Exception as e:
        print(f"Could not list input directory contents: {e}")
    raise FileNotFoundError(f"Data file not found at the specified path: {data_file_path}")
else:
    print("Data file found.")
# --- End Check ---

# Load the dataset using the 'text' loader, passing the exact file path string
# For the 'text' loader, data_files can be a single string path
dataset = load_dataset("text", data_files=data_file_path, split="train")

print("Dataset loaded:")
print(dataset)
# print("\nSample entry:") # Optional
# if len(dataset) > 0:
#     print(dataset[0]['text'][:500])
# else:
#     print("Dataset is empty after loading.")


# --- Dataset Split Section (remains the same) ---
split_seed = 3407
eval_size = 0.05

# Check if dataset is large enough to split
if len(dataset) == 0:
    raise ValueError("Cannot split an empty dataset.")
elif len(dataset) * (1.0 - eval_size) < 1:
     raise ValueError("Dataset is too small to create a non-empty training split.")
elif len(dataset) * eval_size < 1:
     print("Warning: Dataset is very small, evaluation split might have only 1 or few examples.")
     eval_size = 1 / len(dataset) if len(dataset) > 0 else 0.0 # Prevent division by zero

if eval_size > 0:
    print(f"Splitting dataset with eval_size={eval_size:.4f} and seed={split_seed}...")
    dataset_dict = dataset.train_test_split(test_size=eval_size, seed=split_seed, shuffle=True)
    train_dataset = dataset_dict["train"]
    eval_dataset = dataset_dict["test"]
    print("\nDataset split:")
    print(f"Training set size: {len(train_dataset)}")
    print(f"Evaluation set size: {len(eval_dataset)}")
else:
    print("Dataset too small to create an evaluation split. Using full dataset for training.")
    train_dataset = dataset
    eval_dataset = None # No evaluation dataset possible

Looking for data file at: /kaggle/input/book-batch1/cleaned_book_batch1.md
Data file found.


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded:
Dataset({
    features: ['text'],
    num_rows: 24679
})
Splitting dataset with eval_size=0.0500 and seed=3407...

Dataset split:
Training set size: 23445
Evaluation set size: 1234


In [10]:
max_seq_length = 2048 # Or your T4-optimized value

# --- Add Explicit Tokenization ---
print(f"Tokenizing dataset with max_seq_length = {max_seq_length}...")

def tokenize_and_chunk(examples):
    # Tokenize the text field
    tokenized_output = tokenizer(examples["text"], truncation=False) # Don't truncate yet

    # Concatenate all texts
    concatenated_examples = {k: sum(tokenized_output[k], []) for k in tokenized_output.keys()}
    total_length = len(concatenated_examples[list(tokenized_output.keys())[0]])

    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= max_seq_length:
        total_length = (total_length // max_seq_length) * max_seq_length

    # Split by chunks of max_seq_length.
    result = {
        k: [t[i : i + max_seq_length] for i in range(0, total_length, max_seq_length)]
        for k, t in concatenated_examples.items()
    }
    # Create labels (for language modeling, labels are the same as input_ids)
    result["labels"] = result["input_ids"].copy()
    return result

# Apply the function to the datasets
# Use batched=True for speed, remove_columns to get rid of the original 'text' column
# Adjust num_proc based on available CPU cores
num_proc_tok = 8 # Or dataset_num_proc value

tokenized_train_dataset = train_dataset.map(
    tokenize_and_chunk,
    batched=True,
    num_proc=num_proc_tok,
    remove_columns=["text"], # Remove the original text column
)

# Tokenize eval dataset if it exists
if eval_dataset:
    tokenized_eval_dataset = eval_dataset.map(
        tokenize_and_chunk,
        batched=True,
        num_proc=num_proc_tok,
        remove_columns=["text"],
    )
else:
    tokenized_eval_dataset = None

print("Tokenization finished.")
print("Tokenized Training Dataset example:")
print(tokenized_train_dataset)

Tokenizing dataset with max_seq_length = 2048...


Map (num_proc=8):   0%|          | 0/23445 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1234 [00:00<?, ? examples/s]

Tokenization finished.
Tokenized Training Dataset example:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 252
})


In [20]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_train_dataset,
    eval_dataset = tokenized_eval_dataset,
    data_collator = data_collator,
    max_seq_length = max_seq_length, # Use the defined value

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 4,         # Must be 1 for 16GB T4 CPT likely
        gradient_accumulation_steps = 4,        # Effective batch size = 16. Can increase (eg 32) if VRAM allows & want stability.

        warmup_ratio = 0.1,                      # Warmup over 10% of training
        # Choose ONE of max_steps or num_train_epochs
        # Option 1: Max Steps (easier control for CPT)
        max_steps = 10,                      # Adjust based on dataset size & time. Start smaller (eg 500) to test for OOM.
        # Option 2: Epochs (use if you want full passes)
        # num_train_epochs = 1,                # Example: Train for 1 full epoch

        # Lower learning rates for CPT
        learning_rate = 1e-5,                    # Lowered main LR
        embedding_learning_rate = 1e-6,          # Lowered embedding LR (10x ratio)

        # T4 only supports FP16
        fp16 = True,
        bf16 = False,

        logging_steps = 25,                      # Log moderately often
        optim = "paged_adamw_8bit",            # Most memory efficient AdamW variant
        weight_decay = 0.01,                     # Regularization, recommended over 0.00
        lr_scheduler_type = "cosine",            # Good default scheduler

        # Evaluation and Saving strategy (important!)
        # Use 'steps' if using max_steps, 'epoch' if using num_train_epochs
        evaluation_strategy = "steps",           # Match save_strategy below
        eval_steps = 250,                        # Evaluate every 250 steps (adjust frequency)
        save_strategy = "steps",                 # Save based on steps
        save_steps = 500,                        # Save checkpoint every 500 steps (adjust frequency)
        save_total_limit = 2,                    # Keep only last 2 checkpoints + the best one

        load_best_model_at_end = True,           # Important
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        seed = 3407,
        output_dir = "outputs_math_cpt_t4",       # Specific output dir name
        report_to = "none",                      # Change if using WandB/Tensorboard
    ),
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [21]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 252 | Num Epochs = 313 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 614,465,536/5,000,000,000 (12.29% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print("Training finished. Saving final adapter model...")

# Define the output directory in the Kaggle working directory
final_adapter_path = "/kaggle/working/final_math_adapter"

# Save the LoRA adapters
model.save_pretrained(final_adapter_path)

# Optionally, save the tokenizer as well (recommended)
tokenizer.save_pretrained(final_adapter_path)

print(f"Adapter model saved to: {final_adapter_path}")